<a href="https://colab.research.google.com/github/HiddenSquid0622/EDNA/blob/main/EDNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.models import Sequential
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import OneHotEncoder
from tensorflow.keras.utils import to_categorical

In [ ]:
#df = pd.read_csv(r"C:\Users\ASUS\Desktop\EDNA.csv")
from google.colab import files
uploaded = files.upload()

Saving EDNA.csv to EDNA.csv


In [ ]:
 df = pd.read_csv("EDNA.csv")

In [ ]:
print(df.columns)

Index(['Sample_ID', 'Taxonomic_Marker', 'Collection_Date', 'Latitude',
       'Longitude', 'Environment_Broad_Scale', 'Environment_Local_Scale',
       'Environment_Medium', 'Sample_Type', 'Volume_or_Mass', 'Filter_Type',
       'Filter_Pore_Size_um', 'Target_Gene', 'Primers', 'Sequencing_Method',
       'NCBI_BioProject', 'NCBI_BioSample', 'NCBI_SRA_Accession',
       'DNA_Sequence', 'Kingdom', 'Phylum', 'Class', 'Order', 'Family',
       'Genus', 'Species'],
      dtype='object')


In [ ]:
def one_hot_encode_sequence(sequence):
    nucleotide_map = {'A': [1, 0, 0, 0], 'T': [0, 1, 0, 0], 'C': [0, 0, 1, 0], 'G': [0, 0, 0, 1]}
    encoded_sequence = []
    if not isinstance(sequence, str) or len(sequence) == 0:
        return np.zeros((0, 4), dtype=np.float32)
    encoded_sequence = [nucleotide_map.get(nuc, [0, 0, 0, 0]) for nuc in sequence]
    return np.array(encoded_sequence, dtype=np.float32)
df['Encoded_Sequence'] = df['DNA_Sequence'].apply(one_hot_encode_sequence)

In [ ]:
df['Encoded_Sequence'].apply(type).value_counts()

,count
Encoded_Sequence,
<class 'numpy.ndarray'>,5000


In [ ]:
assert all(isinstance(seq, np.ndarray) for seq in df['Encoded_Sequence']), "Some sequences are not arrays"

In [ ]:
def preprocess_dataset(df):
    def convert_volume_mass(value):
        value = str(value).lower().strip()
        if "ml" in value:
            return float(value.replace("ml","").strip())
        if "l" in value:
            return float(value.replace("l","").strip()) * 1000
        if "kg" in value:
            return float(value.replace("kg","").strip()) * 1000
        if "g" in value:
            return float(value.replace("g","").strip())
        try:
            return float(value)
        except:
            return 0.0

    df['Volume_or_Mass'] = df['Volume_or_Mass'].apply(convert_volume_mass)

    numeric_cols = ['Latitude', 'Longitude', 'Volume_or_Mass', 'Filter_Pore_Size_um']
    df.dropna(subset=numeric_cols, inplace=True)

    minmax = MinMaxScaler(feature_range=(-1, 1))
    df[numeric_cols] = minmax.fit_transform(df[numeric_cols])

    return df

df = preprocess_dataset(df)

In [ ]:
numeric_cols = ['Latitude', 'Longitude', 'Volume_or_Mass', 'Filter_Pore_Size_um']
categorical_cols = [col for col in df.columns
                    if df[col].dtype == 'object'
                    and col != 'DNA_Sequence'
                    and all(isinstance(x, (str, float, int)) for x in df[col])]

In [ ]:
def get_numeric_tensor(df, numeric_cols):
    return torch.FloatTensor(df[numeric_cols].values.astype(float))
numeric_tensor = get_numeric_tensor(df, numeric_cols)

In [ ]:
def encode_categorical(df, categorical_cols):
    encoders = {}
    if not categorical_cols:
        return encoders
    protected_cols = ["DNA_Sequence", "Encoded_Sequence"]
    categorical_cols = [col for col in categorical_cols if col not in protected_cols]

    if not categorical_cols:
        return encoders
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded = ohe.fit_transform(df[categorical_cols])

    encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(categorical_cols))

    df.drop(columns=categorical_cols, inplace=True)
    df.reset_index(drop=True, inplace=True)
    df[encoded_df.columns] = encoded_df
    encoders['onehot'] = ohe
    return encoders

In [ ]:
one_hot_cols = [col for col in df.columns if col not in numeric_cols and np.issubdtype(df[col].dtype, np.number)]
cat_tensor = torch.FloatTensor(df[one_hot_cols].values)

In [ ]:
def one_hot_encode_sequence(sequence):
    nucleotide_map = {'A': [1, 0, 0, 0],
                      'T': [0, 1, 0, 0],
                      'C': [0, 0, 1, 0],
                      'G': [0, 0, 0, 1]}
    if not isinstance(sequence, str) or len(sequence) == 0:
        return np.zeros((0, 4), dtype=np.float32)
    encoded = [nucleotide_map.get(nuc, [0, 0, 0, 0]) for nuc in sequence]
    return np.array(encoded, dtype=np.float32)

df['Encoded_Sequence'] = df['DNA_Sequence'].apply(one_hot_encode_sequence)
assert all(isinstance(seq, np.ndarray) for seq in df['Encoded_Sequence']), "Some sequences are not arrays"
max_len = max(seq.shape[0] for seq in df['Encoded_Sequence'])
padded_sequences = pad_sequences(df['Encoded_Sequence'].tolist(),
                                 maxlen=max_len,
                                 padding='post',
                                 dtype='float32')
dna_tensor = torch.FloatTensor(padded_sequences)
dna_tensor_flat = dna_tensor.view(dna_tensor.size(0), -1)

In [ ]:
def create_embeddings(df, categorical_cols, embedding_dims=None):
    embeddings = {}
    if embedding_dims is None:
        embedding_dims = {}
    for col in categorical_cols:
        df[col] = df[col].fillna("UNK")
        df[col + "_idx"], uniques = pd.factorize(df[col])
        num_categories = len(uniques)
        emb_size = embedding_dims.get(col, min(50, (num_categories + 1) // 2))
        embeddings[col] = nn.Embedding(num_categories, emb_size)
    return embeddings

embedding_dims = {col: min(50, (df[col].nunique() + 1) // 2) for col in categorical_cols}

emb_layers = create_embeddings(df, categorical_cols, embedding_dims)


In [ ]:
def combine_features(numeric_tensor, df, categorical_cols, emb_layers, dna_tensor_flat):
    cat_embeds = []
    for col in categorical_cols:
        indices = torch.LongTensor(df[col + "_idx"].values)
        cat_embeds.append(emb_layers[col](indices))
    if cat_embeds:
        cat_embeds_tensor = torch.cat(cat_embeds, dim=1)
        final_tensor = torch.cat([numeric_tensor, cat_embeds_tensor, dna_tensor_flat], dim=1)
    else:
        final_tensor = torch.cat([numeric_tensor, dna_tensor_flat], dim=1)
    return final_tensor


In [ ]:
final_tensor_emb = combine_features(numeric_tensor, df, categorical_cols, emb_layers, dna_tensor_flat)
print( final_tensor_emb.shape)

torch.Size([4167, 1902])


In [ ]:
df['Encoded_Sequence'].apply(type).value_counts()

,count
Encoded_Sequence,
<class 'numpy.ndarray'>,4167


In [ ]:
df['Encoded_Sequence'].apply(type).value_counts()

,count
Encoded_Sequence,
<class 'numpy.ndarray'>,4167


In [ ]:
print("Categorical tensor shape:", cat_tensor.shape)

Categorical tensor shape: torch.Size([4167, 0])


In [ ]:
max_len = max(seq.shape[0] for seq in df['Encoded_Sequence'])
padded_sequences = pad_sequences(
    df['Encoded_Sequence'].tolist(),
    maxlen=max_len,
    dtype='float32',
    padding='post',
    value=0.0
)
seq_array = np.array(padded_sequences, dtype=np.float32)
seq_tensor = torch.from_numpy(seq_array)

In [ ]:
seq_tensor.shape

torch.Size([4167, 380, 4])

In [ ]:
seq_tensor_flat = seq_tensor.view(seq_tensor.size(0), -1)

In [ ]:
#final_tensor = torch.cat([final_tensor, seq_tensor], dim=1)
final_tensor = numeric_tensor
final_tensor = torch.cat([final_tensor, seq_tensor_flat], dim=1)

In [ ]:
print(final_tensor.shape)

torch.Size([4167, 1524])


In [ ]:
len(numeric_cols) + sum(embedding_dims.values())

382

In [ ]:
final_tensor_np = final_tensor.detach().numpy()
input_dim = final_tensor_np.shape[1]

In [ ]:
def build_cnn_classifier(input_dim, num_classes):
    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_dim))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
def build_autoencoder(input_dim, encoding_dim=128, dropout_rate=0.2):

    input_layer = Input(shape=(input_dim,))
    encoded = Dense(512, activation='relu')(input_layer)
    encoded = Dropout(dropout_rate)(encoded)
    encoded = Dense(encoding_dim, activation='relu')(encoded)

    decoded = Dense(512, activation='relu')(encoded)
    decoded = Dense(input_dim, activation='sigmoid')(decoded)

    autoencoder = Model(inputs=input_layer, outputs=decoded)
    encoder = Model(inputs=input_layer, outputs=encoded)

    autoencoder.compile(optimizer='adam', loss='mse')

    return autoencoder, encoder

In [ ]:
def build_cnn_classifier(input_dim, num_classes):

    input_layer = Input(shape=(input_dim, 1))

    x = Conv1D(64, kernel_size=3, activation='relu')(input_layer)
    x = MaxPooling1D(pool_size=2)(x)
    x = Conv1D(128, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    output = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=input_layer, outputs=output)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    return model


In [ ]:
#New Code
input_dim = final_tensor_np.shape[1]
autoencoder, encoder = build_autoencoder(input_dim)
history = autoencoder.fit(
    final_tensor_np, final_tensor_np,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)
encoded_features = encoder.predict(final_tensor_np)
max_len = max(seq.shape[0] for seq in df['Encoded_Sequence'])

padded_sequences = pad_sequences(
    df['Encoded_Sequence'].tolist(),
    maxlen=max_len,
    dtype='float32',
    padding='post',
    value=0.0
)
X = np.array(padded_sequences, dtype=np.float32)
X = X.reshape(X.shape[0], X.shape[1], 4)

label_encoder = LabelEncoder()
y_int = label_encoder.fit_transform(df['Species'])   # strings → integers
y = to_categorical(y_int)                            # integers → one-hot

cnn = build_cnn_classifier(input_shape=(X.shape[1], X.shape[2]), num_classes=y.shape[1])

history = cnn.fit(
    X, y,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - loss: 0.1677 - val_loss: 0.1464
Epoch 2/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 4s 33ms/step - loss: 0.1457 - val_loss: 0.1450
Epoch 3/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - loss: 0.1432 - val_loss: 0.1427
Epoch 4/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - loss: 0.1409 - val_loss: 0.1407
Epoch 5/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - loss: 0.1368 - val_loss: 0.1399
Epoch 6/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - loss: 0.1343 - val_loss: 0.1385
Epoch 7/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 0.1328 - val_loss: 0.1381
Epoch 8/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - loss: 0.1302 - val_loss: 0.1378
Epoch 9/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - loss: 0.1283 - val_loss: 0.1380
Epoch 10/10
105/105 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - loss: 0.1266 - val_loss: 0.1378
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


TypeError: build_cnn_classifier() got an unexpected keyword argument 'input_shape'

In [ ]:
"""
input_dim = final_tensor_np.shape[1]
autoencoder, encoder = build_autoencoder(input_dim)

history = autoencoder.fit(
    final_tensor_np, final_tensor_np,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

encoded_features = encoder.predict(final_tensor_np)


X = np.array(df['Encoded_Sequence'].tolist())
seq_len = len(df['Encoded_Sequence'].iloc[0])
X = X.reshape(X.shape[0], seq_len, 1)

y = keras.utils.to_categorical(df['Species'].values)

cnn = build_cnn_classifier(seq_len, num_classes=y.shape[1])

cnn.fit(
    X, y,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)
"""